# 策略迭代 (Policy Iteration) vs 价值迭代 (Value Iteration) 实战对比

本 Notebook 在经典的 4x4 网格世界中实操对比两种动态规划算法：
- **策略迭代**：策略评估与策略改进交替，观察每一步是否都是完整策略。
- **价值迭代**：贝尔曼最优算子单步迭代，观察价值水波如何从终点反向扩散。

环境说明：4x4 网格，起点为 (0,0)，右下角 (3,3) 为目标宝藏 (+10 分)。

In [1]:
# 【功能模块 1】4x4 网格世界环境建模与奖励设定
# 【作用说明】定义 16 个状态、终点位置 (15)、折扣因子 gamma 及上下左右移动规则
import numpy as np

# 4x4 网格世界基础配置
GRID_H, GRID_W = 4, 4
NUM_STATES = GRID_H * GRID_W
GOAL_STATE = 15  # 右下角 (3, 3)
GAMMA = 0.9      # 折扣因子

# 4个动作：上、下、左、右
ACTIONS = [(-1, 0), (1, 0), (0, -1), (0, 1)]
ACTION_NAMES = ['上', '下', '左', '右']
ACTION_ARROWS = ['↑', '↓', '←', '→']

def get_next_state_and_reward(s, a_idx):
    if s == GOAL_STATE:
        return s, 0.0
    r, c = divmod(s, GRID_W)
    dr, dc = ACTIONS[a_idx]
    nr, nc = max(0, min(GRID_H - 1, r + dr)), max(0, min(GRID_W - 1, c + dc))
    next_s = nr * GRID_W + nc
    reward = 10.0 if next_s == GOAL_STATE else 0.0
    return next_s, reward

def print_policy(policy, V=None):
    print('\n--- 当前策略指向图 ---')
    for r in range(GRID_H):
        row_str = ''
        for c in range(GRID_W):
            s = r * GRID_W + c
            if s == GOAL_STATE:
                row_str += '  🏆  '
            else:
                row_str += f'  {ACTION_ARROWS[policy[s]]}   '
        print(row_str)
    if V is not None:
        print('对应状态价值 V(s)：')
        print(np.round(V.reshape((GRID_H, GRID_W)), 2))
    print('-' * 30)

## 1. 测试【策略迭代】(Policy Iteration)
观察：每一次策略改进后，整张地图上的每个格子是否都有连贯的箭头指向终点。

In [2]:
# 【功能模块 2】策略迭代核心算法 (Policy Iteration)
# 【作用说明】交替执行【策略评估】(算到底直到价值收敛) 与【策略改进】(贪心挑选最大动作)，观察策略一步步变优
def policy_evaluation(policy, V, theta=1e-4):
    """策略评估：固定当前策略，算到底直到收敛"""
    while True:
        delta = 0
        new_V = np.copy(V)
        for s in range(NUM_STATES - 1):
            a = policy[s]
            next_s, r = get_next_state_and_reward(s, a)
            new_V[s] = r + GAMMA * V[next_s]
            delta = max(delta, abs(new_V[s] - V[s]))
        V = new_V
        if delta < theta:
            break
    return V
1li'lun'shang
def policy_iteration():
    V = np.zeros(NUM_STATES)
    # 初始策略：全部盲目向右走
    policy = np.full(NUM_STATES, 3, dtype=int)
    print('【策略迭代开始】初始策略 (全部向右)：')
    print_policy(policy, V)
    
    it = 0
    while True:
        it += 1
        # 1. 策略评估 (把当前打法推演到极限)
        V = policy_evaluation(policy, V)
        
        # 2. 策略改进 (全图贪心更新)
        policy_stable = True
        for s in range(NUM_STATES - 1):
            old_a = policy[s]
            q_values = []
            for a in range(4):
                next_s, r = get_next_state_and_reward(s, a)
                q_values.append(r + GAMMA * V[next_s])
            best_a = int(np.argmax(q_values))
            if best_a != old_a:
                policy_stable = False
            policy[s] = best_a
            
        print(f'===> 第 {it} 次策略迭代完成（每个格子都有明确连通路径）：')
        print_policy(policy, V)
        
        if policy_stable:
            print(f'🎉 策略迭代在仅第 {it} 轮就完美收敛至最优策略！')
            break
    return policy, V

opt_policy, opt_V_pi = policy_iteration()

【策略迭代开始】初始策略 (全部向右)：

--- 当前策略指向图 ---
  →     →     →     →   
  →     →     →     →   
  →     →     →     →   
  →     →     →     🏆  
对应状态价值 V(s)：
[[0. 0. 0. 0.]
 [0. 0. 0. 0.]
 [0. 0. 0. 0.]
 [0. 0. 0. 0.]]
------------------------------
===> 第 1 次策略迭代完成（每个格子都有明确连通路径）：

--- 当前策略指向图 ---
  ↑     ↑     ↑     ↑   
  ↑     ↑     ↑     ↑   
  ↓     ↓     ↓     ↓   
  →     →     →     🏆  
对应状态价值 V(s)：
[[ 0.   0.   0.   0. ]
 [ 0.   0.   0.   0. ]
 [ 0.   0.   0.   0. ]
 [ 8.1  9.  10.   0. ]]
------------------------------
===> 第 2 次策略迭代完成（每个格子都有明确连通路径）：

--- 当前策略指向图 ---
  ↑     ↑     ↑     ↑   
  ↓     ↓     ↓     ↓   
  ↓     ↓     ↓     ↓   
  →     →     →     🏆  
对应状态价值 V(s)：
[[ 0.    0.    0.    0.  ]
 [ 0.    0.    0.    0.  ]
 [ 7.29  8.1   9.   10.  ]
 [ 8.1   9.   10.    0.  ]]
------------------------------
===> 第 3 次策略迭代完成（每个格子都有明确连通路径）：

--- 当前策略指向图 ---
  ↓     ↓     ↓     ↓   
  ↓     ↓     ↓     ↓   
  ↓     ↓     ↓     ↓   
  →     →     →     🏆  
对应状态价值 V(s)：
[[ 0.    0.

## 2. 测试【价值迭代】(Value Iteration)
观察：价值数值是如何一步步从终点向起点倒退扩散（水波反传）的，中间未波及的地方全为 0。

In [3]:
# 【功能模块 3】价值迭代核心算法 (Value Iteration)
# 【作用说明】直接基于贝尔曼最优方程执行单步价值更新，观察价值水波从终点向外倒退扩散的物理过程
def value_iteration(theta=1e-4):
    V = np.zeros(NUM_STATES)
    print('【价值迭代开始】初始价值全部为 0：')
    it = 0
    while True:
        it += 1
        delta = 0
        new_V = np.copy(V)
        for s in range(NUM_STATES - 1):
            q_values = []
            for a in range(4):
                next_s, r = get_next_state_and_reward(s, a)
                q_values.append(r + GAMMA * V[next_s])
            new_V[s] = max(q_values)
            delta = max(delta, abs(new_V[s] - V[s]))
        V = new_V
        
        # 打印前 3 轮的价值水波扩散状态
        if it <= 3 or delta < theta:
            print(f'\n===> 价值迭代第 {it} 步的状态价值表 (注意观察离终点较远的 0 分盲区)：')
            print(np.round(V.reshape((GRID_H, GRID_W)), 2))
            
        if delta < theta:
            print(f'\n🎉 价值迭代在第 {it} 步波纹彻底填满全图，收敛完成！')
            break
            
    # 从收敛的 V 提取最优策略
    policy = np.zeros(NUM_STATES, dtype=int)
    for s in range(NUM_STATES - 1):
        q_values = [get_next_state_and_reward(s, a)[1] + GAMMA * V[get_next_state_and_reward(s, a)[0]] for a in range(4)]
        policy[s] = int(np.argmax(q_values))
    print('\n最终收敛后的最优策略地图：')
    print_policy(policy, V)
    return policy, V

opt_policy_vi, opt_V_vi = value_iteration()

【价值迭代开始】初始价值全部为 0：

===> 价值迭代第 1 步的状态价值表 (注意观察离终点较远的 0 分盲区)：
[[ 0.  0.  0.  0.]
 [ 0.  0.  0.  0.]
 [ 0.  0.  0. 10.]
 [ 0.  0. 10.  0.]]

===> 价值迭代第 2 步的状态价值表 (注意观察离终点较远的 0 分盲区)：
[[ 0.  0.  0.  0.]
 [ 0.  0.  0.  9.]
 [ 0.  0.  9. 10.]
 [ 0.  9. 10.  0.]]

===> 价值迭代第 3 步的状态价值表 (注意观察离终点较远的 0 分盲区)：
[[ 0.   0.   0.   8.1]
 [ 0.   0.   8.1  9. ]
 [ 0.   8.1  9.  10. ]
 [ 8.1  9.  10.   0. ]]

===> 价值迭代第 7 步的状态价值表 (注意观察离终点较远的 0 分盲区)：
[[ 5.9   6.56  7.29  8.1 ]
 [ 6.56  7.29  8.1   9.  ]
 [ 7.29  8.1   9.   10.  ]
 [ 8.1   9.   10.    0.  ]]

🎉 价值迭代在第 7 步波纹彻底填满全图，收敛完成！

最终收敛后的最优策略地图：

--- 当前策略指向图 ---
  ↓     ↓     ↓     ↓   
  ↓     ↓     ↓     ↓   
  ↓     ↓     ↓     ↓   
  →     →     →     🏆  
对应状态价值 V(s)：
[[ 5.9   6.56  7.29  8.1 ]
 [ 6.56  7.29  8.1   9.  ]
 [ 7.29  8.1   9.   10.  ]
 [ 8.1   9.   10.    0.  ]]
------------------------------
